In [1]:
%env NX_CUGRAPH_AUTOCONFIG=True
import networkx as nx
from itertools import combinations
from collections import defaultdict, Counter
#import igraph as ig
import pandas as pd
import sqlite3
import random

env: NX_CUGRAPH_AUTOCONFIG=True



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/software/python-anaconda-2023.09-el8-x86_64/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/software/python-anaconda-2023.09-el8-x86_64/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/software/python-anaconda-2023.09-el8-x86_64/lib/python3.11/site-packages/ipykernel/kernelapp.py

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/software/python-anaconda-2023.09-el8-x86_64/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/software/python-anaconda-2023.09-el8-x86_64/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/software/python-anaconda-2023.09-el8-x86_64/lib/python3.11/site-packages/ipykernel/kernelapp.py

AttributeError: _ARRAY_API not found

In [3]:
conn = sqlite3.connect("tiktok_breadth_first.db")
cursor = conn.cursor()

In [4]:
query = """
SELECT hashtag_names
FROM videos
JOIN follow_relations
ON videos.reposter_username = follow_relations.from_username
WHERE follow_relations.to_username = 'kamalahq'
"""

kamalahq_docs = cursor.execute(query).fetchall()

In [16]:
query = """
SELECT hashtag_names
FROM videos
JOIN follow_relations
ON videos.reposter_username = follow_relations.from_username
WHERE follow_relations.to_username = 'teamtrump'
"""

teamtrump_docs = cursor.execute(query).fetchall()

In [5]:
kamalahq_token = Counter()
for doc in kamalahq_docs:
    contents = doc[0].lower().split(',')
    kamalahq_token.update(contents)

In [25]:
teamtrump_token = Counter()
for doc in teamtrump_docs:
    contents = doc[0].lower().split(',')
    teamtrump_token.update(contents)

In [27]:
kamalahq_edge_weights = defaultdict(int)
for doc in kamalahq_docs:
    contents = doc[0].lower().split(',')
    tokens = sorted(set(contents)) # sorted such that edge weight pairs are indexed alphabetically
    for u, v in combinations(tokens, 2):
        kamalahq_edge_weights[(u, v)] += 1

In [28]:
teamtrump_edge_weights = defaultdict(int)
for doc in teamtrump_docs:
    contents = doc[0].lower().split(',')
    tokens = sorted(set(contents)) # sorted such that edge weight pairs are indexed alphabetically
    for u, v in combinations(tokens, 2):
        teamtrump_edge_weights[(u, v)] += 1

In [ ]:
G_k = nx.Graph()
for (u, v), weight in kamalahq_edge_weights.items():
    G_k.add_edge(u, v, weight=weight)

In [29]:
G_t = nx.Graph()
for (u, v), weight in teamtrump_edge_weights.items():
    G_t.add_edge(u, v, weight=weight)

In [10]:
nx.write_weighted_edgelist(G_k, 'kamalahq_hashtag_network.edgelist')

In [ ]:
nx.write_weighted_edgelist(G_t, 'teamtrump_hashtag_network.edgelist')

In [3]:
G_k = nx.read_weighted_edgelist('kamalahq_hashtag_network.edgelist')

In [4]:
G_t = nx.read_weighted_edgelist('teamtrump_hashtag_network.edgelist')

In [ ]:
import json

In [6]:
k_degree = nx.degree_centrality(G_k)

In [7]:
k_file_path = "k_degree.json"
with open(k_file_path, 'w') as json_file:
    json.dump(k_degree, json_file, indent=4)

In [8]:
t_degree = nx.degree_centrality(G_t)

In [9]:
t_file_path = "t_degree.json"
with open(t_file_path, 'w') as json_file:
    json.dump(t_degree, json_file, indent=4)

In [6]:
random.seed(42)
k_betweenness = nx.betweenness_centrality(G_k, k=50)

In [21]:
k_file_path = "k_betweenness.json"
with open(k_file_path, 'w') as json_file:
    json.dump(k_betweenness, json_file, indent=4)

In [9]:
random.seed(42)
t_betweenness = nx.betweenness_centrality(G_t, k=50)

In [22]:
t_file_path = "t_betweenness.json"
with open(t_file_path, 'w') as json_file:
    json.dump(t_betweenness, json_file, indent=4)

In [10]:
k_closeness = nx.closeness_centrality(G_k, distance='weight')

KeyboardInterrupt: 

In [ ]:
k_file_path = "k_closeness.json"
with open(k_file_path, 'w') as json_file:
    json.dump(k_closeness, json_file, indent=4)

In [ ]:
t_closeness = nx.closeness_centrality(G_t, distance='weight')

In [ ]:
t_file_path = "t_closeness.json"
with open(t_file_path, 'w') as json_file:
    json.dump(t_closeness, json_file, indent=4)

In [ ]:
k_clustering = nx.clustering(G_k, weight='weight')
t_clustering = nx.clustering(G_t, weight='weight')

In [ ]:
k_file_path = "k_clustering.json"
with open(k_file_path, 'w') as json_file:
    json.dump(k_clustering, json_file, indent=4)

In [ ]:
t_file_path = "t_clustering.json"
with open(t_file_path, 'w') as json_file:
    json.dump(t_clustering, json_file, indent=4)